# 07 - Consensus Clustering

**Problem framing.** A single KMeans fit can change with its random seed and with which customers happen to be in the sample. Before acting on a segmentation we want a partition that survives that randomness. Consensus (ensemble) clustering runs many base clusterings on resampled data, records how often each pair of customers is grouped together, and derives the final segments from that agreement.

**Assumptions.** Points that genuinely belong together are grouped together *often*, across seeds and subsamples; transient groupings average out. We assume the number of consensus clusters is roughly known and that a co-association (n x n) matrix fits in memory.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.consensus import build_coassociation_matrix, consensus_clustering
from unsup_lab.data import make_customer_segmentation_data
from unsup_lab.preprocessing import scale_features

## Dataset

The synthetic customer generator has five latent behavioural segments. We scale the features and keep the hidden labels aside for *offline* evaluation only - they are never used during clustering.

In [ ]:
dataset = make_customer_segmentation_data(n_customers=400, random_state=7)
x = scale_features(dataset.features).to_numpy()
truth = dataset.hidden_labels.to_numpy()

def kmeans_factory(seed: int) -> KMeans:
    # Deliberately under-initialised so individual runs are a bit unstable,
    # which is exactly what consensus is meant to absorb.
    return KMeans(n_clusters=5, n_init=1, random_state=seed)

x.shape

## The co-association matrix

Each base run clusters an 80% subsample. The matrix below shows, for every pair of customers, the fraction of runs that placed them in the same cluster (1 = always together).

In [ ]:
coassoc = build_coassociation_matrix(
    x, kmeans_factory, n_runs=40, sample_fraction=0.8, random_state=0
)

# Group similar rows so block structure is visible.
order = coassoc.sum(axis=1).argsort()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(coassoc[order][:, order], cmap="viridis", vmin=0, vmax=1)
ax.set_title("Co-association matrix (reordered)")
ax.set_xlabel("customer")
ax.set_ylabel("customer")
fig.colorbar(im, label="co-clustering frequency")
plt.tight_layout()
plt.show()

## Consensus partition

In [ ]:
result = consensus_clustering(
    x, n_clusters=5, base_factory=kmeans_factory, n_runs=40, random_state=0
)
print(f"consensus stability (mean intra-cluster agreement): {result.stability:.3f}")
consensus_labels = result.labels

## Baseline comparison

We compare the consensus partition against a *single* KMeans fit (the simpler baseline). Agreement with the hidden segments is measured with the Adjusted Rand Index; cohesion with the silhouette score.

In [ ]:
baseline_labels = KMeans(n_clusters=5, n_init=1, random_state=3).fit_predict(x)

comparison = pd.DataFrame(
    {
        "method": ["single KMeans", "consensus"],
        "ARI_vs_hidden": [
            adjusted_rand_score(truth, baseline_labels),
            adjusted_rand_score(truth, consensus_labels),
        ],
        "silhouette": [
            silhouette_score(x, baseline_labels),
            silhouette_score(x, consensus_labels),
        ],
    }
)
comparison.round(3)

## Interpretation

Consensus clustering typically matches or beats a single under-initialised KMeans on agreement with the latent segments, and - more importantly - it comes with a stability score that a single fit cannot provide. The block structure in the co-association heatmap is the visual evidence that the segments are real rather than an initialisation artefact.

## Limitations

- The co-association matrix is ``O(n^2)`` in memory, so this exact form does not scale to very large customer bases without approximation or sampling.
- Consensus stabilises *whatever* the base method tends to find; if every base run shares the same bias (for example KMeans favouring spherical clusters), consensus inherits it.
- The number of consensus clusters is still a modelling choice, not an output.